In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

In [ ]:
df = pd.read_csv("../data/faqs.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df = df.dropna(subset=["question", "answer"])

df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

df = df.drop_duplicates(subset=["question"])

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.reset_index(drop=True)

print("Clean FAQs:", len(df))

In [ ]:
df["text"] = (
    "Question: " + df["question"] +
    "\nAnswer: " + df["answer"]
)

chunks = df["text"].tolist()

print("Number of chunks:", len(chunks))
print("\nExample chunk:\n", chunks[0])

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="faq_collection"
)

embeddings = model.encode(chunks).tolist()

collection.upsert(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings
)

print("Stored FAQs:", collection.count())

In [ ]:
query = "I forgot my password and cannot log in"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

for rank, document in enumerate(
    results["documents"][0],
    start=1
):
    print(f"\nMatch {rank}:")
    print(document)